# Artists data analysis

Let's assert tracks data analysis by taking a first inspection of the artists dataset

In [1]:
import pandas as pd
import altair as alt
from os import path
import warnings
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

#unique user_agent to identify your project to OSM
osm_locator = Nominatim(user_agent="artist_data_analysis_v1", timeout=10) #Nominatim = search engine for OpenStreetMap

#rate-limited to respect OSM's free usage policy
geocode_osm = RateLimiter(osm_locator.geocode, min_delay_seconds=1.0)
reverse_osm = RateLimiter(osm_locator.reverse, min_delay_seconds=1.0)

tqdm.pandas()

dataset_path = path.join('..', 'dataset', 'artists.csv')
df = pd.read_csv(dataset_path, sep=';')

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id_author     104 non-null    object 
 1   name          104 non-null    object 
 2   gender        104 non-null    object 
 3   birth_date    73 non-null     object 
 4   birth_place   72 non-null     object 
 5   nationality   71 non-null     object 
 6   description   86 non-null     object 
 7   active_start  50 non-null     object 
 8   active_end    0 non-null      float64
 9   province      70 non-null     object 
 10  region        68 non-null     object 
 11  country       70 non-null     object 
 12  latitude      72 non-null     float64
 13  longitude     72 non-null     float64
dtypes: float64(3), object(11)
memory usage: 11.5+ KB


validation helper function checking expected types validity

In [3]:
def check_type_validity(value, expected_type):
    return not isinstance(value, expected_type)

before the analysis, let's strip the string columns from any invisible unicode characters:

In [4]:
cols_to_strip = [
    "id_author", "name", "gender", "birth_date",
    "birth_place", "nationality", "description",
    "active_start", "province", "region", "country"
]

zero_width_pattern = r"[\u200b\u200c\u200d\uFEFF]"

nbsp_pattern = r"[\xa0]"

for col in cols_to_strip:
    if col in df.columns:
        #strip only non empty rows
        non_null_mask = df[col].notna()
        
        df.loc[non_null_mask, col] = (
            df.loc[non_null_mask, col]
            .astype(str)
            .str.replace(zero_width_pattern, "", regex=True)
            .str.replace(nbsp_pattern, " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

print("string columns have been stripped and cleaned")

string columns have been stripped and cleaned


### id_author

In [5]:
invalid_elems = df[df['id_author'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['id_author'])

nan_indexes = df.index[df['id_author'].isna()].tolist()
print(f"number of missing values in id_author: {len(nan_indexes)}")

unique_ids = df['id_author'].nunique()
print(f"number of unique id_authors: {unique_ids}")

Series([], Name: id_author, dtype: object)
number of missing values in id_author: 0
number of unique id_authors: 104


as there seems to be no issues related to the id uniqueness of the form "ART{8-digit-number}", a merge between this and the tracks dataset will be performed

In [6]:
tracks_path = path.join('..', 'dataset', 'tracks.csv')
df_tracks = pd.read_csv(tracks_path, sep=',')

merged_df = pd.merge(
    df, 
    df_tracks, 
    left_on='id_author', 
    right_on='id_artist', 
    how='inner'
)

### name

In [7]:
invalid_elems = df[df['name'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['name'])

nan_indexes = df.index[df['name'].isna()].tolist()
print(f"number of missing values in name: {len(nan_indexes)}")

Series([], Name: name, dtype: object)
number of missing values in name: 0


In [8]:
artist_names = merged_df['name'].astype(str).str.lower()
track_artist_names = merged_df['name_artist'].astype(str).str.lower()

mismatches = merged_df[artist_names != track_artist_names]

if mismatches.empty:
    print("all artist names match between the artist andtTracks datasets")
else:
    print(f"found {len(mismatches)} rows where artist names do not match")
    
    unique_mismatch_examples = mismatches.drop_duplicates(subset=['id_author'])
    cols_to_show = ['id_author', 'name', 'name_artist']
    print(unique_mismatch_examples[cols_to_show])

found 872 rows where artist names do not match
        id_author              name     name_artist
344   ART64265460         anna pepe            ANNA
1599  ART67409252  chadia rodriguez          Chadia
2397  ART63985757    dargen d_amico  Dargen D’Amico
4989  ART04141409       guè pequeno             Guè
5960  ART88199433       joey funboy      Joey (ITA)
6994  ART37807199            mike24        Highsnob
7070  ART43601431         miss keta       M¥SS KETA
7630  ART71846481          mr. rain         Mr.Rain
8526  ART42220690            o zulù         ’O Zulù
9509  ART56967402      samuel heron    Samuel Costa


the mismatches are related to the name form, there are no errors. As the column 'name' seems to be more normalized and complete, the name_artist column will be dropped. Rhe only discrepancy is with ART37807199, where the name_artist value (Highsnob) is more 'accurate', as mike24 is a pseudonym of the same person.

### gender

In [9]:
invalid_elems = df[df['gender'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['gender'])

nan_indexes = df.index[df['gender'].isna()].tolist()
print(f"number of missing values in gender: {len(nan_indexes)}")

Series([], Name: gender, dtype: object)
number of missing values in gender: 0


In [10]:
print(df['gender'].value_counts(dropna=False))

gender
M    87
F    17
Name: count, dtype: int64


### birth_date

In [11]:
invalid_elems = df[df['birth_date'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['birth_date'].head(10))

nan_indexes = df.index[df['birth_date'].isna()].tolist()
print(f"number of missing values in birth date: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
17    NaN
Name: birth_date, dtype: object
number of missing values in birth date: 31


In [12]:
dates_converted = pd.to_datetime(df['birth_date'], errors='coerce')

malformed_mask = df['birth_date'].notna() & dates_converted.isna()
malformed_rows = df[malformed_mask]
print(f"number of malformed date strings (cannot be parsed, will be set to NaT): {len(malformed_rows)}")

if not malformed_rows.empty:
    print("malformed dates:")
    print(malformed_rows['birth_date'].unique()[:10])

df['birth_date'] = dates_converted

#checking for syntactically wrong values
now = pd.Timestamp.now()
future_dates = df[df['birth_date'] > now]

if not future_dates.empty:
    print(future_dates[['id_author', 'name', 'birth_date']])

old_date_threshold = pd.Timestamp("1950-01-01")
ancient_dates = df[df['birth_date'] < old_date_threshold]

if not ancient_dates.empty:
    print(ancient_dates[['id_author', 'name', 'birth_date']].head())

number of malformed date strings (cannot be parsed, will be set to NaT): 1
malformed dates:
['http://www.wikidata.org/.well-known/genid/4111f32c49a23235b2e902dc8621d27c']


### birth_place

In [13]:
invalid_elems = df[df['birth_place'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['birth_place'].head(10))

nan_indexes = df.index[df['birth_place'].isna()].tolist()
print(f"number of missing values in birth place: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
17    NaN
Name: birth_place, dtype: object
number of missing values in birth place: 32


In [14]:
unique_places = sorted(df['birth_place'].dropna().unique().tolist())

print(f"number of unique birth places: {len(unique_places)}")
print(unique_places)

number of unique birth places: 40
['Almería', 'Alpignano', 'Avellino', 'Bologna', 'Brescia', 'Buenos Aires', 'Cagliari', 'Desenzano del Garda', 'Firenze', 'Fiumicino', 'Gallarate', 'Genova', 'Grottaglie', 'Grugliasco', 'La Spezia', 'Lodi', 'Milano', 'Napoli', 'Nicosia', 'Nocera Inferiore', 'Olbia', 'Padova', 'Pieve Emanuele', 'Reggio Calabria', 'Rho', 'Roma', 'Salerno', 'San Benedetto del Tronto', 'San Siro', 'Scafati', 'Scampia', 'Senigallia', 'Sesto San Giovanni', 'Singapore', 'Sternatia', 'Torino', 'Treviso', 'Verona', 'Vicenza', 'Vimercate']


### nationality

In [15]:
invalid_elems = df[df['nationality'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['nationality'].head(10))

nan_indexes = df.index[df['nationality'].isna()].tolist()
print(f"number of missing values in nationality: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
17    NaN
Name: nationality, dtype: object
number of missing values in nationality: 33


In [16]:
unique_nationalities = sorted(df['nationality'].dropna().unique().tolist())

print(f"number of unique nationalities: {len(unique_nationalities)}")
print(unique_nationalities)

number of unique nationalities: 2
['Argentina', 'Italia']


### description

In [17]:
invalid_elems = df[df['description'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['description'].head(10))

nan_indexes = df.index[df['description'].isna()].tolist()
print(f"number of missing values in description: {len(nan_indexes)}")

2     NaN
3     NaN
9     NaN
11    NaN
22    NaN
24    NaN
31    NaN
42    NaN
44    NaN
50    NaN
Name: description, dtype: object
number of missing values in description: 18


In [18]:
print(df['description'].head(20))

0                              gruppo musicale italiano
1                  cantautore e rapper italiano (1990-)
2                                                   NaN
3                                                   NaN
4                      gruppo musicale hip hop italiano
5                                     cantante italiano
6                 cantautrice e rapper italiana (1983-)
7     rapper, disc jockey, beatmaker e produttore di...
8                                               cognome
9                                                   NaN
10                                              cognome
11                                                  NaN
12                                              cognome
13    rapper, cantautore e produttore discografico i...
14                              rapper italiano (1998-)
15                              rapper italiana (1998-)
16                     rapper e attore italiano (1982-)
17                             gruppo musicale i

the only relevant information in this column could be the what it looks to be the year of starting rap career activity, so it could be use to fill eventually some missing active_start values.

### active_start

In [19]:
invalid_elems = df[df['active_start'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['active_start'].head(10))

nan_indexes = df.index[df['active_start'].isna()].tolist()
print(f"number of missing values in active start: {len(nan_indexes)}")

2     NaN
3     NaN
5     NaN
8     NaN
10    NaN
11    NaN
12    NaN
14    NaN
15    NaN
16    NaN
Name: active_start, dtype: object
number of missing values in active start: 54


In [20]:
dates_converted = pd.to_datetime(df['active_start'], errors='coerce')

malformed_mask = df['active_start'].notna() & dates_converted.isna()
malformed_rows = df[malformed_mask]
print(f"number of malformed date strings (cannot be parsed, will be set to NaT): {len(malformed_rows)}")

if not malformed_rows.empty:
    print("malformed dates:")
    print(malformed_rows['active_start'].unique()[:10])

df['active_start'] = dates_converted

#checking for syntactically wrong values
now = pd.Timestamp.now()
future_dates = df[df['active_start'] > now]

if not future_dates.empty:
    print(future_dates[['id_author', 'name', 'active_start']])

number of malformed date strings (cannot be parsed, will be set to NaT): 0


as many values are missing, it's probably possible to obtain the year of start rap activity of the artists from the description column. let's first check that the description values are matching the active_start of the row already present:

In [21]:
regex_pattern = r'\((\d{4})'
df['desc_extracted_year'] = df['description'].astype(str).str.extract(regex_pattern)

df['active_start_year'] = df['active_start'].dt.year

valid_comparison_mask = df['desc_extracted_year'].notna() & df['active_start_year'].notna()
compare_df = df.loc[valid_comparison_mask].copy()

compare_df['desc_extracted_year'] = compare_df['desc_extracted_year'].astype(int)

print(f"number of rows with both description-year and active_start available: {len(compare_df)}")

matches = compare_df[compare_df['desc_extracted_year'] == compare_df['active_start_year']]
mismatches = compare_df[compare_df['desc_extracted_year'] != compare_df['active_start_year']]

print(f"matches: {len(matches)}")
print(f"mismatches: {len(mismatches)}")

if not mismatches.empty:
    mismatches['diff_years'] = mismatches['active_start_year'] - mismatches['desc_extracted_year']
    
    cols_to_show = ['desc_extracted_year', 'active_start', 'diff_years']
    print(mismatches[cols_to_show].head(10))

number of rows with both description-year and active_start available: 39
matches: 0
mismatches: 39
    desc_extracted_year active_start  diff_years
1                  1990   2012-01-01        22.0
6                  1983   2007-01-01        24.0
13                 1973   1996-01-01        23.0
18                 1983   2002-01-01        19.0
27                 1989   2006-01-01        17.0
28                 1985   2002-01-01        17.0
30                 1993   2011-01-01        18.0
32                 1976   1996-01-01        20.0
33                 1989   2006-01-01        17.0
34                 1989   2006-01-01        17.0


as the active start year of the column 'description' doesn't match any of the active_start values, then we can say that 'description' contains no useful information, so it can be deleted.

### active_end

In [22]:
invalid_elems = df[df['active_end'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['active_end'].head(10))

nan_indexes = df.index[df['active_end'].isna()].tolist()
print(f"number of missing values in active end: {len(nan_indexes)}")

0   NaN
1   NaN
2   NaN
3   NaN
4   NaN
5   NaN
6   NaN
7   NaN
8   NaN
9   NaN
Name: active_end, dtype: float64
number of missing values in active end: 104


as no one of those artists did stop his/her career, we can safely delete this column as well.

### province

In [23]:
invalid_elems = df[df['province'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['province'].head(10))

nan_indexes = df.index[df['province'].isna()].tolist()
print(f"number of missing values in province: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
6     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
Name: province, dtype: object
number of missing values in province: 34


In [24]:
unique_provinces = sorted(df['province'].dropna().unique().tolist())

print(f"number of unique provinces: {len(unique_provinces)}")
print(unique_provinces)

number of unique provinces: 26
['Ancona', 'Ascoli Piceno', 'Avellino', 'Bologna', 'Brescia', 'Cagliari', 'Enna', 'Firenze', 'Gallura', 'Genova', 'La Spezia', 'Lecce', 'Lodi', 'Milano', 'Monza e della Brianza', 'Napoli', 'Padova', 'Reggio Calabria', 'Roma', 'Salerno', 'Taranto', 'Torino', 'Treviso', 'Varese', 'Verona', 'Vicenza']


### region

In [25]:
invalid_elems = df[df['region'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['region'].head(10))

nan_indexes = df.index[df['region'].isna()].tolist()
print(f"number of missing values in region: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
6     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
Name: region, dtype: object
number of missing values in region: 36


In [26]:
unique_regions = sorted(df['region'].dropna().unique().tolist())

print(f"number of unique provinces: {len(unique_regions)}")
print(unique_regions)

number of unique provinces: 13
['Calabria', 'Campania', 'Emilia-Romagna', 'Lazio', 'Liguria', 'Lombardia', 'Marche', 'Piemonte', 'Puglia', 'Sardegna', 'Sicilia', 'Toscana', 'Veneto']


it's possible to fill the region based on the province, so those rows missing the region but not the province will be filled.

In [27]:
mask_missing_region = df['region'].isna() & df['province'].notna()
print(f"rows missing Region but having province: {mask_missing_region.sum()}")

#unique list of provinces that need a region lookup, for a more general applicable solution
provinces_to_lookup = df.loc[mask_missing_region, 'province'].unique()

def find_region_for_province(province_name):
    try:
        query = f"{province_name}, Italy"
        
        location = geocode_osm(query, addressdetails=True, language='it')
        
        if location and 'address' in location.raw:
            addr = location.raw['address']
            
            # --- DEBUG PRINT: What keys did OSM actually return? ---
            # This is crucial. Maybe it calls it 'region' instead of 'state'?
            # We print this for the first few to avoid spamming, or if state is missing.
            found_state = addr.get('state')
            
            if found_state is None:
                print(f"   [DEBUG] Found '{province_name}' but 'state' key is missing.")
                print(f"           Available keys: {list(addr.keys())}")
            
            return found_state
        else:
            print(f"   [DEBUG] OSM returned NO location for query: '{query}'")
            return None
    except Exception as e:
        print(f"   [ERROR] Exception for '{province_name}': {e}")
        return None

province_region_map = {}

if len(provinces_to_lookup) > 0:
    for prov in tqdm(provinces_to_lookup):
        found_region = find_region_for_province(prov)
        if found_region:
            province_region_map[prov] = found_region

    predicted_regions = df.loc[mask_missing_region, 'province'].map(province_region_map)

    print(f"mapping Dictionary created with {len(province_region_map)} entries.")
    if len(province_region_map) > 0:
        print(f"        Sample: {list(province_region_map.items())[:3]}")
    
    df.loc[mask_missing_region, 'region'] = predicted_regions

remaining_missing = (df['region'].isna() & df['province'].notna()).sum()
print(f"rows filled: {mask_missing_region.sum() - remaining_missing}")

rows missing Region but having province: 2


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.04it/s]

mapping Dictionary created with 1 entries.
        Sample: [('Ancona', 'Marche')]
rows filled: 2


### country

In [28]:
invalid_elems = df[df['country'].apply(check_type_validity, expected_type=str)]
print(invalid_elems['country'].head(10))

nan_indexes = df.index[df['country'].isna()].tolist()
print(f"number of missing values in country: {len(nan_indexes)}")

0     NaN
2     NaN
3     NaN
4     NaN
6     NaN
8     NaN
9     NaN
10    NaN
11    NaN
12    NaN
Name: country, dtype: object
number of missing values in country: 34


In [29]:
unique_countries = sorted(df['country'].dropna().unique().tolist())

print(f"number of unique provinces: {len(unique_countries)}")
print(unique_countries)

number of unique provinces: 1
['Italia']


as this column doesn't provide any useful information in this dataset as well, it can be eliminated.

### latitude

In [30]:
invalid_elems = df[df['latitude'].apply(check_type_validity, expected_type=float)]
print(invalid_elems['latitude'].head(10))

nan_indexes = df.index[df['latitude'].isna()].tolist()
print(f"number of missing values in latitude: {len(nan_indexes)}")

Series([], Name: latitude, dtype: float64)
number of missing values in latitude: 32


### longitude

In [31]:
invalid_elems = df[df['longitude'].apply(check_type_validity, expected_type=float)]
print(invalid_elems['longitude'].head(10))

nan_indexes = df.index[df['longitude'].isna()].tolist()
print(f"number of missing values in longitude: {len(nan_indexes)}")

Series([], Name: longitude, dtype: float64)
number of missing values in longitude: 32


let's check which rows have the combinations of region and province, and latitude and longitude. this because it should be possible to obtain region and province if we have latitude and longitude, and should be possible to optain the latitude and longitute if we have region and province (country is assumed to be Italy).

In [32]:
mask_has_address = (
    (df['province'].notna()) & 
    (df['region'].notna()) & 
    (df['latitude'].isna())
)

mask_has_coords = (
    (df['latitude'].notna()) & 
    (df['longitude'].notna()) & 
    (df['province'].isna())
)

print(f"rows to recover using address -> coordinates: {mask_has_address.sum()}")
print(f"rows to recover using coordinates -> address: {mask_has_coords.sum()}")

rows to recover using address -> coordinates: 0
rows to recover using coordinates -> address: 2


In [33]:
def get_osm_address(row):
    try:
        # Pass coordinates as a string "lat, lon"
        query = f"{row['latitude']}, {row['longitude']}"
        location = reverse_osm(query, language='it')
        
        if location and location.raw.get('address'):
            addr = location.raw['address']
            #OSM mapping: 'state' -> Region, 'county' -> Province
            found_region = addr.get('state')
            found_province = addr.get('county', addr.get('city')) # Fallback to city if county missing
            
            return pd.Series([found_province, found_region])
        return pd.Series([None, None])
    except:
        return pd.Series([None, None])

print("querying OpenStreetMap for missing address details...")
cols = ['province', 'region']
df.loc[mask_has_coords, cols] = df[mask_has_coords].progress_apply(
    get_osm_address, axis=1
).values

attempted_rows = df.loc[mask_has_coords]
success_rows = attempted_rows[attempted_rows['region'].notna()]
print(success_rows[['latitude', 'longitude', 'province', 'region']].head(10))

missing_province_count = df['province'].isna().sum()
missing_region_count = df['region'].isna().sum()

# 2. Print Singular Counts
print(f"rows missing province: {missing_province_count}")
print(f"rows missing region: {missing_region_count}")
print(f"rows missing both (region and province): {min(missing_province_count, missing_region_count)}")

querying OpenStreetMap for missing address details...


100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.01it/s]

     latitude  longitude province          region
6   45.080627   7.670717   Torino        Piemonte
89  44.803741  10.143004    Parma  Emilia-Romagna
rows missing province: 32
rows missing region: 32
rows missing both (region and province): 32
